# Week 22: Multimodal Foundation Models & Vision-Language Architectures
## Notebook 02: Cross-Attention in Diffusion & Multimodal Models from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In Self-Attention, Queries ($Q$), Keys ($K$), and Values ($V$) originate from the exact same sequence.
In **Cross-Attention**:
- Queries ($Q$) come from the **Target Modality** (e.g. Image Latent Patches in Stable Diffusion).
- Keys ($K$) and Values ($V$) come from the **Conditioning Modality** (e.g. Text Embeddings from CLIP/T5).
- Every pixel/latent in the image attends to the text description: *"Should this patch become a dog or a cat?"*

**Core Engine of Stable Diffusion, Midjourney, and Multimodal Decoders!**

---

### 2. Mathematical Foundations: Cross-Attention
Given image latent tokens $X_{\text{img}} \in \mathbb{R}^{N_{\text{img}} \times D}$ and text tokens $X_{\text{text}} \in \mathbb{R}^{N_{\text{text}} \times D_t}$:

$$Q = X_{\text{img}} W_Q, \quad K = X_{\text{text}} W_K, \quad V = X_{\text{text}} W_V$$

Cross-Attention Matrix:
$$A = \text{Softmax}\left( \frac{Q K^T}{\sqrt{d}} \right) \in \mathbb{R}^{N_{\text{img}} \times N_{\text{text}}}$$

Notice the rectangular shape ($N_{\text{img}} \times N_{\text{text}}$):
Entry $A_{i, j}$ represents how strongly image patch $i$ attends to text token $j$!

Output:
$$\text{CrossAttention}(X_{\text{img}}, X_{\text{text}}) = A \cdot V \in \mathbb{R}^{N_{\text{img}} \times D}$$

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Define Inputs
# Image Latents: 4 spatial patches (2x2 grid)
N_img = 4
D_img = 6
X_img = np.random.randn(N_img, D_img)

# Text Conditioning: 3 text tokens ["a", "golden", "retriever"]
N_text = 3
D_text = 8
X_text = np.random.randn(N_text, D_text)

# Step 2: Projection Weight Matrices
d_k = 4 # Common projection attention dimension
W_Q = np.random.randn(D_img, d_k) * 0.3
W_K = np.random.randn(D_text, d_k) * 0.3
W_V = np.random.randn(D_text, D_img) * 0.3

# Step 3: Compute Cross-Attention
# Queries from Image, Keys & Values from Text!
Q = X_img @ W_Q       # (4, 4)
K = X_text @ W_K      # (3, 4)
V = X_text @ W_V      # (3, 6)

# Scaled Dot-Product Attention: Q @ K.T (4, 3)
scores = (Q @ K.T) / np.sqrt(d_k)
# Softmax across text tokens (axis=1)
exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attention_weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Output: Attention @ V (4, 6)
out_cross = attention_weights @ V

print("--- Cross-Attention Weight Matrix (Image Patches x Text Tokens) ---")
print("Tokens: ['a', 'golden', 'retriever']")
for patch_idx in range(N_img):
    print(f"  Image Patch {patch_idx}: {np.round(attention_weights[patch_idx], 3)} (Sum = {np.sum(attention_weights[patch_idx]):.1f})")

print(f"\nOutput Feature Shape: {out_cross.shape} (Maintains Image Spatial Structure!)")

### 3. Step-by-Step Code Walkthrough
1. **Asymmetric Input Dimensions:** Image tokens ($D=6$) and text tokens ($D=8$) have different dimensions, successfully bridged into $d_k=4$ attention space.
2. **Text Conditioning Matrix:** The $4 \times 3$ attention map governs how textual concepts map into image coordinates.
3. **Spatial Preservation:** The output retains the exact spatial patch resolution $(4, 6)$ of the image, injecting text guidance into U-Net layers.

---

### 4. Industry Applications & Trade-offs
- **Stable Diffusion U-Net:** Cross-attention layers are inserted in the downsampling and upsampling blocks of the latent U-Net.
- **ControlNet:** Injects additional spatial control conditioning (depth maps, Canny edges) via cross-attention.

---

### 5. Top Interview Questions & Answers
**Q1: What is the fundamental difference between Self-Attention and Cross-Attention?**
> **Answer:** In Self-Attention, $Q, K, V$ are projected from the same sequence, capturing intra-modality relationships ($O(N^2)$ square attention matrix). In Cross-Attention, $Q$ comes from one sequence (e.g. image) and $K, V$ come from another sequence (e.g. text), capturing inter-modality conditioning ($O(N \times M)$ rectangular attention matrix).

**Q2: What is Prompt-to-Prompt editing in diffusion models?**
> **Answer:** In diffusion models, the cross-attention maps determine *where* each text word appears in the generated image. Prompt-to-Prompt editing replaces or injects cross-attention maps during reverse diffusion steps, allowing localized edits (e.g. changing "a cat on a bicycle" to "a dog on a bicycle") while keeping the background intact.